# Cavity Parametric Optimisation and Machine-Learning Surrogate Analysis

This notebook analyses COMSOL parametric sweep data for the cavity design.

It performs four tasks:

1. read the simulation workbook;
2. convert one-parameter sweeps into a machine-learning feature matrix;
3. train surrogate models for resonant frequency and \(V_{m,\mathrm{eff}}\);
4. visualise the design space using PCA, feature importance, and smooth valley maps.

The machine-learning result is intended as a **screening tool** for choosing the next COMSOL simulations. It should not be treated as a fully validated optimum, because the data mainly come from separate one-parameter sweeps rather than a full multi-parameter grid.

## 1. Expected input data

The input Excel workbook should contain the first two sheets for thesis tables or notes:

```text
Sheet 1: Method
Sheet 2: Results
```

These two sheets are ignored by the machine-learning part.

All following sheets should contain COMSOL parametric sweep data. Each sweep sheet should use this layout:

| Column | Meaning | Unit |
|---|---|---|
| A | Varied parameter | m for geometry, unitless for \(\varepsilon_r\) |
| B | Resonant frequency | GHz |
| C | Total magnetic energy, IntW | J |
| D | Maximum magnetic energy density, MaxW | J/m\(^3\) |
| E | Magnetic energy in gain region, GainW | J |
| F | Gain-region volume, GainV | m\(^3\) |
| G | Effective magnetic mode volume, \(V_{m,\mathrm{eff}}\) | m\(^3\) |

The notebook recognises these sheet names by default:

```text
Gap_Width(eps=8)
dielectric(a=1.2mm)
Gap_Length(eps=9,a=1mm)
Metal_thickness
Dielectric_length
Dielectric_width
Dielectric_thickness
```

If your sheet names change, edit `SHEET_CONFIG` in Section 3.

In [ ]:
# Optional: install packages if needed.
# Uncomment and run this cell only if your environment is missing packages.

# !pip install numpy pandas matplotlib scikit-learn scipy openpyxl

## 2. Imports and plotting settings

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.preprocessing import StandardScaler

try:
    from scipy.interpolate import RBFInterpolator
    SCIPY_RBF_AVAILABLE = True
except Exception:
    SCIPY_RBF_AVAILABLE = False


def set_plot_style() -> None:
    # Compact journal-style plotting format.
    plt.rcParams.update({
        "font.family": "DejaVu Sans",
        "font.size": 9,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 8,
        "axes.linewidth": 0.9,
        "lines.linewidth": 1.1,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


set_plot_style()

## 3. Configuration

Edit the file name and baseline geometry here.

The baseline values are used to fill the non-varied parameters in one-parameter sweeps. For example, in the metal-thickness sweep, only \(t_\mathrm{Cu}\) changes, so the other parameters are filled with the baseline values.

In [ ]:
@dataclass
class Config:
    # Change this if your file has a different name.
    input_excel: str | Path = "Simulation Data.xlsx"

    # All figures and processed files will be saved here.
    output_dir: str | Path = "figures_ml"

    # Frequency target used in the objective function.
    target_frequency_ghz: float = 6.46

    # Frequency penalty scale in GHz.
    # Smaller value means stronger penalty for frequency mismatch.
    frequency_scale_ghz: float = 0.10

    # Random seed for reproducibility.
    random_state: int = 3

    # Reference geometry used to fill non-varied parameters.
    # Units are mm, except eps_r.
    baseline: Dict[str, float] = field(default_factory=lambda: {
        "a_mm": 1.0,       # gap width
        "eps_r": 9.0,      # dielectric constant
        "b_mm": 4.092,     # gap length
        "tcu_mm": 0.6,     # metal thickness
        "c_mm": 1.6,       # dielectric length
        "d_mm": 7.09,      # dielectric width
        "tsap_mm": 0.6,    # dielectric thickness
    })


FEATURE_NAMES = [
    "a_mm",
    "eps_r",
    "b_mm",
    "tcu_mm",
    "c_mm",
    "d_mm",
    "tsap_mm",
]


# Sheet-level metadata.
# scale converts the raw spreadsheet value into the feature unit.
# For geometry sweeps, COMSOL values are usually in m, so scale = 1e3 converts m to mm.
SHEET_CONFIG = {
    "Gap_Width(eps=8)": {
        "parameter_label": "Gap width",
        "parameter_symbol": "a",
        "unit": "mm",
        "feature": "a_mm",
        "scale": 1e3,
        "fixed": {"eps_r": 8.0},
    },
    "dielectric(a=1.2mm)": {
        "parameter_label": "Dielectric constant",
        "parameter_symbol": "eps_r",
        "unit": "",
        "feature": "eps_r",
        "scale": 1.0,
        "fixed": {"a_mm": 1.2},
    },
    "Gap_Length(eps=9,a=1mm)": {
        "parameter_label": "Gap length",
        "parameter_symbol": "b",
        "unit": "mm",
        "feature": "b_mm",
        "scale": 1e3,
        "fixed": {"eps_r": 9.0, "a_mm": 1.0},
    },
    "Metal_thickness": {
        "parameter_label": "Metal thickness",
        "parameter_symbol": "t_Cu",
        "unit": "mm",
        "feature": "tcu_mm",
        "scale": 1e3,
        "fixed": {"eps_r": 9.0, "a_mm": 1.0},
    },
    "Dielectric_length": {
        "parameter_label": "Dielectric length",
        "parameter_symbol": "c",
        "unit": "mm",
        "feature": "c_mm",
        "scale": 1e3,
        "fixed": {"eps_r": 9.0, "a_mm": 1.0},
    },
    "Dielectric_width": {
        "parameter_label": "Dielectric width",
        "parameter_symbol": "d",
        "unit": "mm",
        "feature": "d_mm",
        "scale": 1e3,
        "fixed": {"eps_r": 9.0, "a_mm": 1.0},
    },
    "Dielectric_thickness": {
        "parameter_label": "Dielectric thickness",
        "parameter_symbol": "t_sap",
        "unit": "mm",
        "feature": "tsap_mm",
        "scale": 1e3,
        "fixed": {"eps_r": 9.0, "a_mm": 1.0},
    },
}

## 4. Data loading functions

These functions convert the Excel workbook into a clean table. Each row corresponds to one COMSOL simulation.

The objective function is

\[
J =
\log_{10}\left(\frac{V_{m,\mathrm{eff}}}{10^{-11}}\right)
+
\left(\frac{|f-f_\mathrm{target}|}{0.10}\right)^2.
\]

A lower objective means a better compromise between small effective mode volume and frequency matching.

In [ ]:
def clean_numeric(value) -> float:
    # Convert a spreadsheet value to float; return NaN if conversion fails.
    try:
        return float(value)
    except Exception:
        return np.nan


def load_parametric_workbook(config: Config) -> pd.DataFrame:
    # Load COMSOL parametric sweeps from an Excel workbook.
    input_excel = Path(config.input_excel)
    if not input_excel.exists():
        raise FileNotFoundError(
            f"Workbook not found: {input_excel}. "
            "Put the Excel file in the same folder as this notebook, "
            "or edit Config.input_excel."
        )

    all_sheets = pd.read_excel(input_excel, sheet_name=None, engine="openpyxl")
    sheet_names = list(all_sheets.keys())

    rows: List[dict] = []

    # Ignore first two sheets: usually Method and Results.
    for sheet_name in sheet_names[2:]:
        if sheet_name not in SHEET_CONFIG:
            print(f"Skipping unrecognised sheet: {sheet_name}")
            continue

        spec = SHEET_CONFIG[sheet_name]
        df = all_sheets[sheet_name].copy()

        if df.empty or df.shape[1] < 7:
            print(f"Skipping incomplete sheet: {sheet_name}")
            continue

        for _, row in df.iterrows():
            raw_param = clean_numeric(row.iloc[0])
            frequency = clean_numeric(row.iloc[1])
            intW = clean_numeric(row.iloc[2])
            maxW = clean_numeric(row.iloc[3])
            gainW = clean_numeric(row.iloc[4])
            gainV = clean_numeric(row.iloc[5])
            vm_eff = clean_numeric(row.iloc[6])

            if np.isnan(raw_param) or np.isnan(frequency) or np.isnan(vm_eff):
                continue

            features = dict(config.baseline)

            # Convert raw parameter into the feature unit.
            parameter_value = raw_param * spec["scale"]
            features[spec["feature"]] = parameter_value

            # Apply known fixed values for this sweep.
            features.update(spec["fixed"])

            freq_error = abs(frequency - config.target_frequency_ghz)
            objective = (
                np.log10(vm_eff / 1e-11)
                + (freq_error / config.frequency_scale_ghz) ** 2
            )

            rows.append({
                "sheet": sheet_name,
                "parameter_label": spec["parameter_label"],
                "parameter_symbol": spec["parameter_symbol"],
                "unit": spec["unit"],
                "parameter_value": parameter_value,
                "frequency_GHz": frequency,
                "freq_error_GHz": freq_error,
                "IntW_J": intW,
                "MaxW_J_m3": maxW,
                "GainW_J": gainW,
                "GainV_m3": gainV,
                "Vm_eff_m3": vm_eff,
                "log10_Vm_eff": np.log10(vm_eff),
                "objective": objective,
                **features,
            })

    if not rows:
        raise ValueError("No valid simulation rows were extracted. Check sheet names and columns.")

    return pd.DataFrame(rows)


def summarise_dataset(data: pd.DataFrame) -> pd.DataFrame:
    # Return a compact summary of extracted rows by sheet.
    return (
        data.groupby("sheet")
        .agg(
            rows=("sheet", "size"),
            f_min=("frequency_GHz", "min"),
            f_max=("frequency_GHz", "max"),
            vm_min=("Vm_eff_m3", "min"),
            vm_max=("Vm_eff_m3", "max"),
        )
        .reset_index()
    )

## 5. Load the dataset

Run this cell first after changing the file name if needed.

In [ ]:
config = Config(
    input_excel="Simulation Data.xlsx",  # change to "Thesis Table.xlsx" if needed
    output_dir="figures_ml",
    target_frequency_ghz=6.46,
)

output_dir = Path(config.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

data = load_parametric_workbook(config)

print(f"Rows extracted: {len(data)}")
display(summarise_dataset(data))

data.head()

## 6. Train machine-learning surrogate models

Two models are trained:

1. a frequency model predicting \(f\);
2. a mode-volume model predicting \(\log_{10}(V_{m,\mathrm{eff}})\).

The model used is **Extra Trees Regressor**, a tree-based ensemble algorithm. It is suitable here because the dataset is small, nonlinear, and comes from separate parametric sweeps.

In [ ]:
def train_surrogate_models(data: pd.DataFrame, config: Config) -> dict:
    # Train frequency and mode-volume surrogate models.
    X = data[FEATURE_NAMES].to_numpy(float)
    y_freq = data["frequency_GHz"].to_numpy(float)
    y_logv = data["log10_Vm_eff"].to_numpy(float)

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    X_train, X_test, yf_train, yf_test, yv_train, yv_test = train_test_split(
        X_scaled,
        y_freq,
        y_logv,
        test_size=0.25,
        random_state=config.random_state,
    )

    frequency_model = ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=config.random_state,
    )

    vm_model = ExtraTreesRegressor(
        n_estimators=500,
        min_samples_leaf=2,
        random_state=config.random_state + 1,
    )

    frequency_model.fit(X_train, yf_train)
    vm_model.fit(X_train, yv_train)

    yf_pred = frequency_model.predict(X_test)
    yv_pred = vm_model.predict(X_test)

    kfold = KFold(n_splits=5, shuffle=True, random_state=config.random_state)

    metrics = {
        "frequency_r2_test": r2_score(yf_test, yf_pred),
        "frequency_mae_test_GHz": mean_absolute_error(yf_test, yf_pred),
        "vm_r2_test": r2_score(yv_test, yv_pred),
        "vm_mae_test_log10": mean_absolute_error(yv_test, yv_pred),
        "frequency_r2_cv_mean": cross_val_score(
            frequency_model, X_scaled, y_freq, cv=kfold, scoring="r2"
        ).mean(),
        "vm_r2_cv_mean": cross_val_score(
            vm_model, X_scaled, y_logv, cv=kfold, scoring="r2"
        ).mean(),
    }

    # Refit on all data before producing design-space maps.
    frequency_model.fit(X_scaled, y_freq)
    vm_model.fit(X_scaled, y_logv)

    return {
        "scaler": scaler,
        "X_scaled": X_scaled,
        "frequency_model": frequency_model,
        "vm_model": vm_model,
        "metrics": metrics,
        "test_data": {
            "yf_test": yf_test,
            "yf_pred": yf_pred,
            "yv_test": yv_test,
            "yv_pred": yv_pred,
        },
    }


models = train_surrogate_models(data, config)

for key, value in models["metrics"].items():
    print(f"{key}: {value:.4g}")

## 7. ML diagnostic plot

This plot compares COMSOL simulation outputs against model predictions. Good agreement means the surrogate model captures the main trends in the available dataset.

In [ ]:
def plot_diagnostics(models: dict, output_dir: Path) -> None:
    # Plot predicted versus simulated values for both ML models.
    set_plot_style()
    test = models["test_data"]

    fig, axes = plt.subplots(1, 2, figsize=(8.8, 3.2), constrained_layout=True)

    ax = axes[0]
    ax.scatter(test["yf_test"], test["yf_pred"], s=22, alpha=0.8)
    mn = min(test["yf_test"].min(), test["yf_pred"].min())
    mx = max(test["yf_test"].max(), test["yf_pred"].max())
    ax.plot([mn, mx], [mn, mx], "--", color="0.35", linewidth=0.9)
    ax.set_xlabel("Simulated frequency (GHz)")
    ax.set_ylabel("ML-predicted frequency (GHz)")
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)
    ax.text(
        0.04, 0.94,
        f"(a)\n$R^2$={models['metrics']['frequency_r2_test']:.3f}\n"
        f"MAE={models['metrics']['frequency_mae_test_GHz']:.3f} GHz",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    ax = axes[1]
    ax.scatter(test["yv_test"], test["yv_pred"], s=22, color="#9467bd", alpha=0.8)
    mn = min(test["yv_test"].min(), test["yv_pred"].min())
    mx = max(test["yv_test"].max(), test["yv_pred"].max())
    ax.plot([mn, mx], [mn, mx], "--", color="0.35", linewidth=0.9)
    ax.set_xlabel(r"Simulated $\log_{10}(V_{m,\mathrm{eff}})$")
    ax.set_ylabel(r"ML-predicted $\log_{10}(V_{m,\mathrm{eff}})$")
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)
    ax.text(
        0.04, 0.94,
        f"(b)\n$R^2$={models['metrics']['vm_r2_test']:.3f}\n"
        f"MAE={models['metrics']['vm_mae_test_log10']:.3f}",
        transform=ax.transAxes,
        ha="left",
        va="top",
    )

    fig.savefig(output_dir / "ml_surrogate_diagnostics.png", dpi=600, bbox_inches="tight")
    fig.savefig(output_dir / "ml_surrogate_diagnostics.pdf", bbox_inches="tight")
    plt.show()


plot_diagnostics(models, output_dir)

## 8. Feature importance

Feature importance indicates which parameters the model uses most strongly to predict frequency and \(V_{m,\mathrm{eff}}\). This helps connect the machine-learning result to the physical design.

In [ ]:
def plot_feature_importance(models: dict, output_dir: Path) -> pd.DataFrame:
    # Plot and return feature importance for the two surrogate models.
    set_plot_style()

    readable_labels = {
        "a_mm": "Gap width",
        "eps_r": "Dielectric constant",
        "b_mm": "Gap length",
        "tcu_mm": "Metal thickness",
        "c_mm": "Dielectric length",
        "d_mm": "Dielectric width",
        "tsap_mm": "Dielectric thickness",
    }

    freq_importance = models["frequency_model"].feature_importances_
    vm_importance = models["vm_model"].feature_importances_

    importance = pd.DataFrame({
        "feature": FEATURE_NAMES,
        "label": [readable_labels[f] for f in FEATURE_NAMES],
        "frequency_importance": freq_importance,
        "vm_importance": vm_importance,
    })

    labels = importance["label"].to_list()
    x = np.arange(len(labels))
    width = 0.36

    fig, ax = plt.subplots(figsize=(7.2, 3.5), constrained_layout=True)
    ax.bar(x - width / 2, freq_importance, width=width, label="Frequency model")
    ax.bar(x + width / 2, vm_importance, width=width, label=r"$V_{m,\mathrm{eff}}$ model")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel("Feature importance")
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)
    ax.legend(frameon=False, loc="upper right")

    fig.savefig(output_dir / "ml_feature_importance.png", dpi=600, bbox_inches="tight")
    fig.savefig(output_dir / "ml_feature_importance.pdf", bbox_inches="tight")
    plt.show()

    return importance


importance = plot_feature_importance(models, output_dir)
display(importance)

## 9. PCA visualisation

PCA reduces the seven design parameters into two coordinates. The colour represents \(\log_{10}(V_{m,\mathrm{eff}})\). The red star marks the best observed objective value.

In [ ]:
def run_pca(data: pd.DataFrame, X_scaled: np.ndarray) -> tuple[PCA, pd.DataFrame]:
    # Run PCA and append PC1/PC2 to the dataframe.
    pca = PCA(n_components=2, random_state=0)
    coords = pca.fit_transform(X_scaled)

    data_pca = data.copy()
    data_pca["PC1"] = coords[:, 0]
    data_pca["PC2"] = coords[:, 1]

    return pca, data_pca


def plot_pca(data_pca: pd.DataFrame, pca: PCA, output_dir: Path) -> None:
    # Plot PCA coordinates coloured by log10(Vm_eff).
    set_plot_style()

    fig, ax = plt.subplots(figsize=(5.8, 4.2), constrained_layout=True)

    scatter = ax.scatter(
        data_pca["PC1"],
        data_pca["PC2"],
        c=data_pca["log10_Vm_eff"],
        cmap="viridis",
        s=22,
        edgecolor="none",
        alpha=0.85,
    )

    best = data_pca.loc[data_pca["objective"].idxmin()]
    ax.scatter(
        best["PC1"],
        best["PC2"],
        marker="*",
        s=130,
        color="red",
        edgecolor="white",
        linewidth=0.6,
    )

    ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0] * 100:.1f}%)")
    ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1] * 100:.1f}%)")
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)

    cbar = fig.colorbar(scatter, ax=ax)
    cbar.set_label(r"$\log_{10}(V_{m,\mathrm{eff}})$")

    fig.savefig(output_dir / "ml_pca_design_space.png", dpi=600, bbox_inches="tight")
    fig.savefig(output_dir / "ml_pca_design_space.pdf", bbox_inches="tight")
    plt.show()


pca, data_pca = run_pca(data, models["X_scaled"])
plot_pca(data_pca, pca, output_dir)

print("PCA explained variance:")
print(f"PC1: {pca.explained_variance_ratio_[0]:.3f}")
print(f"PC2: {pca.explained_variance_ratio_[1]:.3f}")

## 10. Smooth surrogate valley map

This section produces a smooth guide map using RBF interpolation. The map shows a predicted low-objective region in the gap-width / metal-thickness plane.

This is useful for selecting the next COMSOL simulation, but it should not be treated as the final optimum. The input data are mainly one-parameter sweeps, so the smooth map includes interpolation and some extrapolation.

In [ ]:
def make_grid_from_baseline(config: Config, a_values: np.ndarray, tcu_values: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    # Create a physical feature grid over gap width and metal thickness.
    A, T = np.meshgrid(a_values, tcu_values)

    X_grid = np.zeros((A.size, len(FEATURE_NAMES)))
    for i, name in enumerate(FEATURE_NAMES):
        X_grid[:, i] = config.baseline[name]

    X_grid[:, FEATURE_NAMES.index("a_mm")] = A.ravel()
    X_grid[:, FEATURE_NAMES.index("tcu_mm")] = T.ravel()
    X_grid[:, FEATURE_NAMES.index("eps_r")] = 9.0

    return A, T, X_grid


def plot_smooth_valley_map(data: pd.DataFrame, models: dict, config: Config, output_dir: Path) -> Optional[dict]:
    # Plot a smooth RBF surrogate map over gap width and metal thickness.
    if not SCIPY_RBF_AVAILABLE:
        print("SciPy RBFInterpolator is not available. Skipping RBF map.")
        return None

    set_plot_style()

    X_scaled = models["X_scaled"]
    scaler = models["scaler"]

    y_freq = data["frequency_GHz"].to_numpy(float)
    y_logv = data["log10_Vm_eff"].to_numpy(float)

    rbf_freq = RBFInterpolator(
        X_scaled,
        y_freq,
        kernel="gaussian",
        epsilon=1.4,
        smoothing=0.015,
        degree=-1,
        neighbors=100,
    )

    rbf_logv = RBFInterpolator(
        X_scaled,
        y_logv,
        kernel="gaussian",
        epsilon=1.4,
        smoothing=0.015,
        degree=-1,
        neighbors=100,
    )

    # Region near the current cavity design.
    a_values = np.linspace(0.02, 1.20, 300)
    tcu_values = np.linspace(0.10, 3.00, 300)

    A, T, X_grid = make_grid_from_baseline(config, a_values, tcu_values)
    X_grid_scaled = scaler.transform(X_grid)

    F = rbf_freq(X_grid_scaled).reshape(A.shape)
    LOGV = rbf_logv(X_grid_scaled).reshape(A.shape)
    VM = 10 ** LOGV

    OBJ = (
        np.log10(VM / 1e-11)
        + (np.abs(F - config.target_frequency_ghz) / config.frequency_scale_ghz) ** 2
    )

    best_idx = np.unravel_index(np.nanargmin(OBJ), OBJ.shape)

    best = {
        "a_mm": float(A[best_idx]),
        "tcu_mm": float(T[best_idx]),
        "frequency_GHz": float(F[best_idx]),
        "Vm_eff_m3": float(VM[best_idx]),
        "objective": float(OBJ[best_idx]),
    }

    def robust_levels(Z, n=100):
        zmin = float(np.nanmin(Z))
        zmax = float(np.nanmax(Z))
        if np.isclose(zmin, zmax):
            zmax = zmin + 1e-9
        return np.linspace(zmin, zmax, n)

    fig, ax = plt.subplots(figsize=(6.2, 4.8), constrained_layout=True)

    contour = ax.contourf(
        A,
        T,
        OBJ,
        levels=robust_levels(OBJ),
        cmap="viridis",
        extend="neither",
    )

    if np.nanmin(F) <= config.target_frequency_ghz <= np.nanmax(F):
        freq_contour = ax.contour(
            A,
            T,
            F,
            levels=[config.target_frequency_ghz],
            colors="red",
            linewidths=1.6,
        )
        ax.clabel(
            freq_contour,
            fmt={config.target_frequency_ghz: f"{config.target_frequency_ghz:.2f} GHz"},
            fontsize=7,
        )

    ax.scatter(
        best["a_mm"],
        best["tcu_mm"],
        marker="*",
        s=145,
        color="red",
        edgecolor="white",
        linewidth=0.6,
        label="Predicted valley",
    )

    ax.scatter(
        config.baseline["a_mm"],
        config.baseline["tcu_mm"],
        marker="o",
        s=45,
        color="white",
        edgecolor="black",
        linewidth=0.5,
        label="Reference",
    )

    ax.set_xlabel(r"Gap width, $a$ (mm)")
    ax.set_ylabel(r"Metal thickness, $t_\mathrm{Cu}$ (mm)")
    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)
    ax.legend(frameon=False, loc="best")

    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_label("Smoothed ML objective")

    fig.savefig(output_dir / "smooth_gap_metal_thickness_valley_map.png", dpi=600, bbox_inches="tight")
    fig.savefig(output_dir / "smooth_gap_metal_thickness_valley_map.pdf", bbox_inches="tight")
    plt.show()

    return best


valley_candidate = plot_smooth_valley_map(data, models, config, output_dir)
valley_candidate

## 11. Identify useful candidate points

This section lists the main candidate designs:

1. lowest objective;
2. closest frequency match;
3. smallest effective mode volume;
4. predicted smooth-valley point.

In [ ]:
def collect_candidates(data: pd.DataFrame, valley_candidate: Optional[dict]) -> pd.DataFrame:
    # Collect the main candidate points into a table.
    best_objective = data.loc[data["objective"].idxmin()]
    closest_frequency = data.loc[data["freq_error_GHz"].idxmin()]
    smallest_vm = data.loc[data["Vm_eff_m3"].idxmin()]

    rows = [
        {
            "candidate": "Best observed objective",
            "sheet": best_objective["sheet"],
            "parameter": best_objective["parameter_label"],
            "value": best_objective["parameter_value"],
            "unit": best_objective["unit"],
            "frequency_GHz": best_objective["frequency_GHz"],
            "Vm_eff_m3": best_objective["Vm_eff_m3"],
            "objective": best_objective["objective"],
            "comment": "Best balance of frequency matching and mode-volume reduction in observed data.",
        },
        {
            "candidate": "Closest observed frequency",
            "sheet": closest_frequency["sheet"],
            "parameter": closest_frequency["parameter_label"],
            "value": closest_frequency["parameter_value"],
            "unit": closest_frequency["unit"],
            "frequency_GHz": closest_frequency["frequency_GHz"],
            "Vm_eff_m3": closest_frequency["Vm_eff_m3"],
            "objective": closest_frequency["objective"],
            "comment": "Closest simulated point to target frequency.",
        },
        {
            "candidate": "Smallest observed Vm_eff",
            "sheet": smallest_vm["sheet"],
            "parameter": smallest_vm["parameter_label"],
            "value": smallest_vm["parameter_value"],
            "unit": smallest_vm["unit"],
            "frequency_GHz": smallest_vm["frequency_GHz"],
            "Vm_eff_m3": smallest_vm["Vm_eff_m3"],
            "objective": smallest_vm["objective"],
            "comment": "Smallest mode volume, but may be far from the target frequency.",
        },
    ]

    if valley_candidate is not None:
        rows.append({
            "candidate": "Predicted smooth-valley point",
            "sheet": "RBF surrogate map",
            "parameter": "a, t_Cu",
            "value": f"{valley_candidate['a_mm']:.3f}, {valley_candidate['tcu_mm']:.3f}",
            "unit": "mm",
            "frequency_GHz": valley_candidate["frequency_GHz"],
            "Vm_eff_m3": valley_candidate["Vm_eff_m3"],
            "objective": valley_candidate["objective"],
            "comment": "Suggested next COMSOL simulation; not a final validated optimum.",
        })

    return pd.DataFrame(rows)


candidates = collect_candidates(data, valley_candidate)
display(candidates)

candidates.to_csv(output_dir / "candidate_summary.csv", index=False)

## 12. Save processed data and model summary

This makes the analysis easier to reproduce and upload to GitHub.

In [ ]:
# Save processed feature matrix.
data_pca.to_csv(output_dir / "processed_ml_dataset.csv", index=False)

# Save feature importance.
importance.to_csv(output_dir / "feature_importance.csv", index=False)

# Save model summary.
summary = {
    "rows_used": int(len(data)),
    "target_frequency_ghz": config.target_frequency_ghz,
    "frequency_scale_ghz": config.frequency_scale_ghz,
    "metrics": {k: float(v) for k, v in models["metrics"].items()},
    "pca_explained_variance": {
        "PC1": float(pca.explained_variance_ratio_[0]),
        "PC2": float(pca.explained_variance_ratio_[1]),
    },
    "valley_candidate": valley_candidate,
}

with open(output_dir / "summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print(f"Files saved in: {output_dir.resolve()}")
for path in sorted(output_dir.iterdir()):
    print(path.name)

## 13. Short interpretation for thesis writing

Suggested wording:

> Machine learning was used as a surrogate analysis because the full \(Q\)-factor and multi-parameter optimisation were limited by simulation cost. Accurate \(Q\)-factor evaluation requires finer mesh refinement near metal surfaces, slot edges, and dielectric interfaces, which increases the computational time. Therefore, the available one-parameter sweeps were combined into a feature matrix containing gap width, dielectric constant, gap length, metal thickness, and dielectric dimensions. PCA was used to visualise the design space, and an Extra Trees Regressor was trained to predict resonant frequency and \(\log_{10}(V_{m,\mathrm{eff}})\).

> The model reproduced the COMSOL trends well and identified gap width as the dominant parameter for both frequency and effective mode volume. This confirms that the slot geometry is the main control for magnetic-field localisation. Metal thickness and dielectric constant provide secondary tuning, which can help recover the target resonance frequency after the field has been localised. The smooth valley map therefore provides a useful guide for selecting the next COMSOL simulations, but the predicted optimum should be verified directly because the training data are based mainly on separate one-parameter sweeps.